### 연습 문제
- Doc2Vec를 이용하여 감성 분석
- 데이터는 ratings_train.txt 파일을 로드
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화 함수를 이용
    - document 컬럼의 데이터에서 중복 데이터를 제거
    - 빈 텍스트가 존재한다면 해당 데이터 제거
    - 상위 5000개 데이터를 학습 데이터로 이용
- 토큰화 함수는 Komoran을 사용
    - 품사 필터 : NNP, NNG, VV, VA, MAG, XR 만을 사용
    - 불용어 단어 : 하다, 되다, 이다, 것, 수, 거 단어들을 제외
- 독립 변수(document), 종속 변수(label) 데이터를 나눠주고 train,test 데이터로 분할(8:2)
- Doc2Vec 객체를 생성하여 벡터화
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습 데이터는 train 데이터를 이용
- Doc2Vec 객체애서 train, test 데이터를 infer_vector() 함수를 이용하여 벡터 데이터를 생성
- ML 분류 모델을 이용하여 임베딩된 데이터를 독립변수로 사용하여 학습
    - 정확도를 확인
    - Logistic
        - max_iter = 2000
        - random_state = 42
    - LinearSVC
        - random_state = 42
- test데이터를 이용하여 2개의 모델 중 정확도 높은 모델을 검색

In [81]:
import pandas as pd
import numpy as np
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
# from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import re

In [53]:
df=pd.read_csv('./ratings_train.txt', sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [54]:
df.dropna(inplace=True)

In [55]:
#정규화 함수
def normalize(text):
    #특수 문자, 공백에 대한 처리
    text=re.sub(r'[^가-핳0-0a-xA-z\s\.]',' ',str(text))
    text=re.sub(r'\s+',' ',text).strip()
    return text

In [56]:
#document 컬럼의 데이터를 정규화
df['document']=df['document'].map(normalize)

In [57]:
df2=df.copy()

#실수가 많은 부분 -> df2가 DataFrame에서 Series로 바뀜
df2=df2['document'].map(normalize)
df2

0                                       아 더빙.. 진짜 짜증나네요 목소리
1                          ...포스터보고 초딩영 줄....오버연기조차 가볍지 않구나
2                                         너무재밓었다그래서보는것을추천한다
3                              교도소 이야기구먼 ..솔직 재미는 없다..평점 조정
4         사이몬페그의 익살스런 연기가 돋보였던 영 스파이더맨에서 늙어보이기만 던 커스틴 던스...
                                ...                        
149995                                  인간이 문제지.. 소는 뭔죄인가..
149996                                        평점이 너무 낮아서...
149997                          이게 뭐요 한국인은 거들먹거리고 필리핀 은 착하다
149998                          청춘 영 의 최고봉.방 과 우울 던 날들의 자 상
149999                               한국 영 최초로 수간하는 내용이 담긴 영
Name: document, Length: 149995, dtype: str

In [60]:
#공백 데이터기 존재하는가?존재하면 정규화를 통해서 ' ' -> '' 작업 후 strip을 사용하하면 ''
# df[df['document']=='']
df=df.loc[
    ~(df['document']==''),
]

In [61]:
df.drop_duplicates('document',inplace=True)

In [62]:
komoran=Komoran()

#특정 품사 사용
allow_pos=['NNP','NNG','VV','VA','MAG','XR']
#불용어 처리
stop_word=['하다','되다','이다','것','수','거']

def tokenize(text):
    tokens=[]
    for word, pos in komoran.pos(text):
        if pos in allow_pos and word not in stop_word:
                tokens.append(word)
    return tokens

In [63]:
df3=df.head(5000)

In [ ]:
df3.iloc[2]

In [65]:
tokenize_sentence=[
    tokenize(text) for text in df3['document'].values
]
tokenize_sentence

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영', '스파이더맨', '늙', '보이', '덜', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리', '못', '다'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '정말',
  '발',
  '도',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영'],
 ['왜', '평점', '낮', '꽤', '보', '우드', '너무', '길들이', '있'],
 [],
 ['볼', '때', '눈물', '나서', '죽', '0년대', '자극', '늘', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '단보', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽', '못'],
 ['담백', '깔끔', '서', '좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['은',
  '존중',
  '진짜',
  '극장',
  '보',
  '영',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이돌',
  '

In [67]:
X=tokenize_sentence
y=df3['label'].values

In [68]:
#train_test 데이터셋을 분할
X_train,X_test,y_train,y_test=train_test_split(
    X, y, test_size=0.2, stratify=y
)
df3['label'].value_counts()

label
0    2504
1    2496
Name: count, dtype: int64

In [69]:
#문서에 Tag 부착
#빈 토큰 리스트를 제외하고 태그를 부착
def tagged_docs(token_data):
    tagged=[]
    for idx, toks in enumerate(token_data):
        #toks의 길이가 0이라면 학습에서 큰 의미가 없음 -> 제외
        if len(toks) == 0:
            continue

        tagged.append(
            TaggedDocument(
                words=toks, tags=[f'DOC_{idx}']
            )
        )
    return tagged

In [70]:
X_train_tag=tagged_docs(X_train)
print(len(X_train_tag), ' | ', len(X_train))

3926  |  4000


In [72]:
#Doc2Vec 객체를 생성
model=Doc2Vec(
    documents=X_train_tag,
    vector_size=200,
    window=5,
    min_count=2,
    dm=1,
    epochs=50,
    seed=42,
    negative=5  #잘못된 단어 배치를 사용하여 학습에서 이용
)

In [74]:
#Doc2Vec 객체를 생성하고 추후에 학습
model2=Doc2Vec(
    vector_size=200,
    window=5,
    min_count=2,
    dm=1,
    epochs=50,
    seed=42,
    negative=5  
)
#단어 사전 생성
model2.build_vocab(X_train_tag)
#학습
model2.train(
    X_train_tag, total_examples=len(X_train_tag), epochs=50
)

In [75]:
#2개의 모델에서 단어 사전의 개수를 확인
print(len(model.wv))
print(len(model2.wv))

2649
2649


In [76]:
print(len(model.dv))
print(len(model2.dv))

3926
3926


In [79]:
def infer_vector(model, norm_texts, epochs=50):
    #norm_texts : 텍스트 정규화가 끝나고 토큰화가 완료된 데이터
    #model : 임베딩 모델

    result=[]

    for tokens in norm_texts:
        #tokens의 길이가 0이라면 0 행렬로 되돌려준다.
        if len(tokens) == 0 :
            result.append(
                np.zeros(model.vector_size, dtype = np.float32)
            )
        else:
            vec=model.infer_vector(tokens, epochs=epochs)
            result.append(vec)
    # return np.array(result)
    return np.vstack(result)

In [80]:
X_train_vec=infer_vector(model, X_train)
X_test_vec=infer_vector(model, X_test)


X_train_vec.shape

In [82]:
def eval_clf(model, X_train, X_test, y_train, y_test):
    #model : 분류 모델 입력
    model.fit(X_train, y_train)

    pred=model.predict(X_test)
    print(classification_report(pred,y_test))

In [83]:
logi=LogisticRegression(max_iter=2000, random_state=42)
svc=LinearSVC(random_state=42)

eval_clf(logi,X_train_vec, X_test_vec, y_train, y_test)
eval_clf(svc,X_train_vec, X_test_vec, y_train, y_test)

              precision    recall  f1-score   support

           0       0.77      0.75      0.76       512
           1       0.75      0.76      0.76       488

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000

              precision    recall  f1-score   support

           0       0.77      0.76      0.77       507
           1       0.76      0.77      0.76       493

    accuracy                           0.76      1000
   macro avg       0.76      0.76      0.76      1000
weighted avg       0.76      0.76      0.76      1000



In [84]:
#DataFrame을 train, test로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_df['label'].value_counts()

label
0    58186
1    57546
Name: count, dtype: int64

In [85]:
test_df['label'].value_counts()

label
0    14547
1    14387
Name: count, dtype: int64